# 04 — Baseline RF/XGB sobre features combinados (EPIC 4)

**Avance 3 — Baseline · CRISP-ML(Q) fase 3 Modeling**

Este notebook entrena dos modelos tabulares (Random Forest y XGBoost) sobre el vector de features del EPIC 3 (AlphaEarth 64-dim + indices espectrales + estadisticas temporales + SRTM + ERA5) y documenta su desempeno contra el umbral minimo del Avance 3.

| Seccion | Contenido | US |
|---------|-----------|-----|
| 1 | Setup y carga del dataset | US-019 |
| 2 | Justificacion del algoritmo (40 pts) | US-019 |
| 3 | Importancia de features nativa | US-020 |
| 4 | Analisis SHAP | US-020 |
| 5 | Conclusiones de feature engineering | US-020 |
| 5b | Curvas de aprendizaje y validacion | US-021 |
| 6 | Desempeno minimo vs umbral 0.60 (10 pts) | US-019 |
| 7 | Comparativa AlphaEarth vs Sentinel-2 crudo | US-022 |
| 8 | Discusion y decisiones para EPIC 5 | US-022 |


## 1. Setup y carga del dataset

El dataset de entrada es el subset PASTIS-R a nivel parcela generado en el EPIC 3 (US-018): 85.951 parcelas x 187 features espectro-temporales. Las etiquetas son las 20 clases de cultivo de PASTIS-R (se descartan las clases de fondo).

In [1]:
# Parametros papermill (celda con tag 'parameters'; sobreescribibles
# en CI con valores reducidos via `papermill -p`).
FEATURES_PATH = 'data/test_fixtures/feature_selection_parcels_subset.parquet'
MAX_SAMPLES = 0  # 0 = dataset completo; >0 = submuestreo estratificado
TUNE = True
F1_THRESHOLD = 0.60


In [2]:
# Parameters
MAX_SAMPLES = 3000
TUNE = False


In [3]:
import warnings

import matplotlib

matplotlib.use('Agg')  # backend headless para papermill/CI
import matplotlib.pyplot as plt
import polars as pl

warnings.filterwarnings('ignore')


In [4]:
from ml.train.baseline import _load_baseline_dataset, _prepare_dataframe

df_raw = _load_baseline_dataset(FEATURES_PATH)
df = _prepare_dataframe(df_raw)
print(f'Parcelas: {df.height:,}  |  Columnas: {df.width}')
df.head()

Parcelas: 85,951  |  Columnas: 192


parcel_id,year,NDVI_mean,NDVI_std,NDVI_min,NDVI_max,NDVI_p05,NDVI_p25,NDVI_p50,NDVI_p75,NDVI_p95,NDWI_mean,NDWI_std,NDWI_min,NDWI_max,NDWI_p05,NDWI_p25,NDWI_p50,NDWI_p75,NDWI_p95,EVI_mean,EVI_std,EVI_min,EVI_max,EVI_p05,EVI_p25,EVI_p50,EVI_p75,EVI_p95,NDMI_mean,NDMI_std,NDMI_min,NDMI_max,NDMI_p05,NDMI_p25,NDMI_p50,NDMI_p75,…,NDVI_fft_amp_0,NDVI_fft_phase_0,NDVI_fft_amp_1,NDVI_fft_phase_1,NDVI_fft_amp_2,NDVI_fft_phase_2,NDVI_fft_amp_3,NDVI_fft_phase_3,NDWI_fft_amp_0,NDWI_fft_phase_0,NDWI_fft_amp_1,NDWI_fft_phase_1,NDWI_fft_amp_2,NDWI_fft_phase_2,NDWI_fft_amp_3,NDWI_fft_phase_3,EVI_fft_amp_0,EVI_fft_phase_0,EVI_fft_amp_1,EVI_fft_phase_1,EVI_fft_amp_2,EVI_fft_phase_2,EVI_fft_amp_3,EVI_fft_phase_3,sog_doy,peak_doy,peak_value,senescence_doy,ndvi_auc,ndvi_slope_pre_peak,ndvi_slope_post_peak,maturity_duration_days,patch_id,instance_id,class_id,fold,n_pixels
str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,f64,i64,f64,f64,f64,i64,i64,i64,i64,i64,i64
"""10000_1""",2018,0.451583,0.425313,-0.068311,2.303833,-0.02314,0.202424,0.30881,0.765301,0.987349,-0.4298,0.338675,-1.603543,0.203464,-0.822717,-0.656512,-0.421072,-0.231754,0.03352,0.282466,0.307185,-0.622353,0.848309,-0.13101,0.14015,0.189091,0.59712,0.721938,0.227215,0.304057,-0.235981,1.0,-0.225231,0.013784,0.166098,0.470995,…,0.406117,0.0,0.116952,0.959278,0.268396,-0.812251,0.099299,-0.671697,0.379628,0.0,0.096763,-1.491115,0.180904,2.22084,0.094876,1.680415,0.250078,0.0,0.096473,1.985061,0.109415,-1.297251,0.089628,-0.86289,null,26,2.303833,34,158.423406,null,-0.258612,5,10000,1,2,1,101
"""10000_2""",2018,0.499859,0.613426,-0.035138,3.852548,0.009661,0.16877,0.265388,0.779707,0.98985,-0.461037,0.451878,-2.664785,0.215764,-0.86394,-0.699827,-0.379332,-0.237233,0.039321,0.288321,0.351033,-1.282691,0.865093,-0.051102,0.124069,0.202867,0.622151,0.701521,0.260024,0.301983,-0.225433,1.0,-0.18454,0.048057,0.26025,0.490585,…,0.447687,0.0,0.119995,0.652629,0.360659,-0.887526,0.154474,-1.078157,0.406423,0.0,0.082969,-1.678393,0.250675,2.142489,0.150508,1.643163,0.263809,0.0,0.164197,2.412146,0.146805,-1.176721,0.016652,0.282209,null,26,3.852548,35,174.858654,null,-0.421472,4,10000,2,2,1,146
"""10000_3""",2018,0.334038,0.27952,-0.019577,1.0,-0.004163,0.14065,0.238016,0.552242,0.834303,-0.349186,0.319539,-0.8,1.030717,-0.777078,-0.604582,-0.335123,-0.228834,0.006794,0.213473,0.190922,-0.104255,0.618046,-0.023873,0.099734,0.138956,0.312178,0.585605,0.11725,0.250841,-0.331999,1.0,-0.162837,-0.064601,0.093857,0.233324,…,0.286841,0.0,0.174465,0.561766,0.163503,0.806981,0.101583,2.325232,0.298671,0.0,0.057297,-1.618627,0.054383,-2.954037,0.122988,0.383855,0.184555,0.0,0.09192,0.972171,0.106464,1.453878,0.081433,2.637839,184,366,1.0,377,111.748535,0.003212,-0.064142,24,10000,3,12,1,222
"""10000_5""",2018,0.38247,0.285132,-0.074386,1.0,0.022891,0.177489,0.318098,0.622669,0.806941,-0.394311,0.262345,-0.940892,0.184607,-0.788802,-0.603789,-0.385081,-0.232968,-0.023878,0.253364,0.197264,-0.182055,0.790065,0.052423,0.12536,0.178894,0.403373,0.580462,0.168672,0.248691,-0.272999,1.0,-0.209728,0.018661,0.152897,0.327187,…,0.35871,0.0,0.112893,2.013443,0.226917,-0.746715,0.054041,0.988047,0.360391,0.0,0.120399,-0.82425,0.168244,2.240376,0.037988,1.114151,0.240716,0.0,0.132097,2.513825,0.143179,-1.109168,0.019577,0.949213,42,366,1.0,377,139.890532,0.000525,-0.065199,5,10000,5,2,1,161
"""10000_7""",2018,0.31712,0.238965,0.001322,1.0,0.041599,0.180184,0.251987,0.3889,0.841654,-0.349337,0.234502,-1.0,0.264819,-0.67687,-0.491825,-0.353776,-0.195299,-0.025525,0.211746,0.151538,-0.036628,0.83954,0.041436,0.119197,0.188436,0.25756,0.496483,0.113138,0.267155,-0.26795,1.079542,-0.19123,-0.040898,0.078001,0.13962,…,0.277805,0.0,0.059054,1.143065,0.100011,-0.166592,0.098765,-0.294571,0.308596,0.0,0.077307,-0.308286,0.

In [5]:
# Distribucion de clases — PASTIS-R tiene desbalance fuerte.
class_counts = (
    df.group_by('class_id').len().sort('len', descending=True)
)
class_counts

class_id,len
i64,u32
1,31292
3,13123
8,10640
2,8206
14,3174
…,…
6,908
9,871
17,848


## 2. Justificacion del algoritmo

Se eligen **Random Forest** y **XGBoost** como baseline tabular. Cuatro argumentos sustentan la decision:

**(a) AlphaEarth ya codifica la informacion multisensor.** El embedding AlphaEarth Foundations de 64 dimensiones condensa informacion optica, radar y temporal aprendida por un Foundation Model entrenado sobre todo el archivo Sentinel. Sobre una representacion ya rica, un modelo tabular es un baseline suficiente y honesto — no se requiere una arquitectura profunda para establecer el lower bound (cf. Brown et al., 2025, *AlphaEarth Foundations*; EDA US-013).

**(b) RF y XGBoost son interpretables.** Ambos exponen importancia de features nativa (Gini para RF, gain para XGBoost) y son compatibles con SHAP (TreeExplainer exacto). El criterio 'Caracteristicas importantes' del Avance 3 (US-020) depende de esta interpretabilidad — un baseline opaco no permitiria auditar que features aportan (Lundberg & Lee, 2017, *SHAP*).

**(c) Robustez a outliers y a la escala.** Los arboles particionan el espacio por umbrales y no asumen ninguna distribucion de las features; outliers residuales tras la winsorizacion del EPIC 3 no desplazan las fronteras de decision como lo harian en un modelo lineal o en una red sin normalizacion cuidadosa.

**(d) Bajo costo computacional.** El problema (85k x 187, 20 clases) se entrena en minutos. XGBoost usa el GPU local cuando esta disponible y degrada a CPU de forma transparente; RF corre siempre en CPU multinucleo. El baseline es reproducible en la laptop de cualquier integrante del equipo y en CI sin reservar computo cloud, dejando el presupuesto H100 para EPIC 5/6.

## 3. Importancia de features nativa

Random Forest y XGBoost exponen una medida de importancia de features sin coste adicional: **Gini/MDI** para RF (`feature_importances_`) y **gain** para XGBoost (`Booster.get_score(importance_type='gain')`). Es el primer diagnostico de interpretabilidad — barato y directo — antes del analisis SHAP de la seccion 4.

Se cargan los **modelos production** persistidos por US-019 (`artifacts/baseline_{rf,xgb}_v1.joblib`); si los artefactos no existen el notebook entrena RF/XGB in-notebook con los hiperparametros base (fallback D8 del plan US-020).

In [6]:
import joblib
from pathlib import Path

from ml.train.baseline import train_one_model

REPORTS_DIR = Path('reports/baseline')
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS = {'rf': Path('artifacts/baseline_rf_v1.joblib'),
             'xgb': Path('artifacts/baseline_xgb_v1.joblib')}

models = {}
for kind, path in ARTIFACTS.items():
    if path.exists():
        payload = joblib.load(path)
        models[kind] = {
            'model': payload['model'],
            'feature_cols': tuple(payload['feature_cols']),
            'source': 'joblib US-019',
        }
    else:
        res = train_one_model(df, model=kind)
        models[kind] = {
            'model': res.model,
            'feature_cols': res.feature_cols,
            'source': 'fallback in-notebook (D8)',
        }
    print(f"{kind.upper()}: {models[kind]['source']}  |  "
          f"{len(models[kind]['feature_cols'])} features")

RF: joblib US-019  |  185 features

XGB: joblib US-019  |  185 features


In [7]:
from ml.eval.interpretability import feature_importance_table

importance = {}
for kind, bundle in models.items():
    table = feature_importance_table(
        bundle['model'], kind, bundle['feature_cols']
    )
    importance[kind] = table
    table.write_csv(REPORTS_DIR / f'feature_importance_{kind}.csv')
importance['rf'].head(10)

2026-05-22 07:42:52 [info     ] feature_importance_table_computed model_kind=rf n_features=185 top_feature=EVI_fft_phase_2


2026-05-22 07:42:52 [info     ] feature_importance_table_computed model_kind=xgb n_features=185 top_feature=MSAVI2_min


feature,importance,rank
str,f64,i64
"""EVI_fft_phase_2""",0.031686,1
"""CCCI_p75""",0.025438,2
"""EVI_fft_phase_1""",0.024245,3
"""EVI_fft_phase_3""",0.020832,4
"""NDCI_p50""",0.019787,5
"""MCARI_p95""",0.019077,6
"""NDVI_fft_phase_2""",0.018781,7
"""NDVI_p50""",0.018475,8
"""NDWI_fft_phase_1""",0.018447,9


In [8]:
# Barplot top-20 de la importancia nativa por modelo.
for kind, table in importance.items():
    top20 = table.head(20)
    fig, ax = plt.subplots(figsize=(8, 6), dpi=200)
    ax.barh(top20['feature'].to_list()[::-1],
            top20['importance'].to_list()[::-1],
            color='#2c7fb8')
    ax.set_xlabel('Importancia (' + ('Gini' if kind == 'rf' else 'gain') + ')')
    ax.set_title(f'Importancia nativa top-20 — {kind.upper()}')
    fig.tight_layout()
    fig.savefig(REPORTS_DIR / f'importance_{kind}_top20.png',
                dpi=200, bbox_inches='tight')
    plt.show()

## 4. Analisis SHAP

La importancia nativa de la seccion 3 ordena los features pero no explica *como* cada uno desplaza la prediccion. **SHAP** (Lundberg & Lee, 2017) descompone cada prediccion en contribuciones aditivas por feature con garantias teoricas de consistencia. Para modelos de arboles se usa `TreeExplainer` (algoritmo TreeSHAP exacto, CPU).

**Detalles de implementacion** (decisiones del plan US-020):

- **D6 — subsample**: SHAP corre sobre un subsample estratificado de ~3.000 parcelas, no sobre las ~85k del dataset; TreeSHAP es exacto pero O(samples x trees x depth).
- **D3 — multiclase**: PASTIS-R tiene 18-20 clases; `compute_shap_values` normaliza la salida multiclase de SHAP (lista-por-clase / array 3D) a un tensor `(n_samples, n_features, n_classes)`.
- **D4 — ranking global**: la importancia global SHAP es la media de `|SHAP|` sobre clases y muestras.

In [9]:
from ml.eval.interpretability import (
    compute_shap_values,
    shap_summary_plot,
    shap_dependence_plots,
    shap_waterfall_plot,
)

SHAP_SAMPLE_SIZE = 3000
shap_results = {}
for kind, bundle in models.items():
    shap_results[kind] = compute_shap_values(
        bundle['model'], df, kind,
        feature_cols=bundle['feature_cols'],
        sample_size=SHAP_SAMPLE_SIZE,
    )
    print(f'{kind.upper()}: tensor SHAP '
          f'{shap_results[kind].values.shape}')

2026-05-22 07:50:01 [info     ] shap_values_computed           model_kind=rf n_classes=18 n_features=185 n_samples=3000 sample_rows=3000


RF: tensor SHAP (3000, 185, 18)


2026-05-22 07:50:06 [info     ] shap_values_computed           model_kind=xgb n_classes=18 n_features=185 n_samples=3000 sample_rows=3000


XGB: tensor SHAP (3000, 185, 18)


In [10]:
# Summary plot (beeswarm/bar) de las top-20 features globales.
for kind, result in shap_results.items():
    fig = shap_summary_plot(result, df, top_n=20)
    fig.savefig(REPORTS_DIR / f'shap_summary_{kind}.png',
                dpi=200, bbox_inches='tight')
    plt.show()

In [11]:
# Dependence plots de los 5 features mas importantes (RF).
dependence = shap_dependence_plots(
    shap_results['rf'], df, top_features=5
)
for idx, (feature_name, fig) in enumerate(dependence, start=1):
    fig.savefig(
        REPORTS_DIR / f'shap_dependence_{idx}_{feature_name}.png',
        dpi=200, bbox_inches='tight',
    )
    plt.show()

2026-05-22 07:50:07 [info     ] shap_dependence_plots_generated class_idx=0 model_kind=rf n_plots=5


In [12]:
# Waterfall de una prediccion ejemplo por modelo.
for kind, result in shap_results.items():
    fig = shap_waterfall_plot(result, row=0)
    fig.savefig(REPORTS_DIR / f'shap_waterfall_{kind}.png',
                dpi=200, bbox_inches='tight')
    plt.show()

### 4.1 Dominancia de las dimensiones AlphaEarth

Pregunta clave para el Paper Track: de las features mas influyentes segun SHAP, **¿cuantas son dimensiones del embedding AlphaEarth** (`dim_00..dim_63`) frente a indices espectrales, estadisticas temporales o bloques contextuales (S1/SRTM/ERA5)? `alphaearth_dominance_table` clasifica cada feature en su familia de origen y cuantifica la dominancia.

In [13]:
from ml.eval.interpretability import alphaearth_dominance_table

dominance = alphaearth_dominance_table(
    shap_results['rf'].global_importance, top_n=20
)
dominance.write_csv(REPORTS_DIR / 'alphaearth_dominance.csv')
dominance

2026-05-22 07:50:08 [info     ] alphaearth_dominance_computed  dominance_ratio=0.0 n_alphaearth=0 top_n=20


rank,feature,family,importance
i64,str,str,f64
1,"""EVI_fft_phase_2""","""spectral_index""",0.004859
2,"""CCCI_p75""","""spectral_index""",0.004565
3,"""EVI_fft_phase_1""","""spectral_index""",0.004053
4,"""MCARI_p95""","""spectral_index""",0.003342
5,"""NDVI_fft_phase_2""","""spectral_index""",0.003161
…,…,…,…
16,"""NDRE_p50""","""spectral_index""",0.002208
17,"""GCVI_p50""","""spectral_index""",0.002186
18,"""EVI_p95""","""spectral_index""",0.002094


In [14]:
# Conteo por familia y conclusion cuantificada (AC-4).
family_counts = (
    dominance.group_by('family').len()
    .sort('len', descending=True)
)
n_alphaearth = int(
    dominance.filter(pl.col('family') == 'alphaearth').height
)
top_ae = (
    dominance.filter(pl.col('family') == 'alphaearth')['feature']
    .to_list()[:3]
)
print(f'{n_alphaearth}/20 de las top features SHAP son '
      f'dimensiones AlphaEarth.')
if top_ae:
    print(f'Lideran: ' + ', '.join(top_ae))
family_counts

0/20 de las top features SHAP son dimensiones AlphaEarth.


family,len
str,u32
"""spectral_index""",20


## 5. Conclusiones de feature engineering

Esta seccion **valida o refuta** las decisiones de Feature Engineering del EPIC 3 (US-018) cruzando los rankings de interpretabilidad de este notebook con los artefactos de seleccion de features:

- `reports/feature_selection/feature_importance_rf.csv` — importancia exploratoria de US-018.
- `reports/feature_selection/anova_f_scores.csv` — F-scores univariados de la seleccion.
- `reports/feature_selection/selected_features.json` — el conjunto que US-018 retuvo.

El objetivo es responder tres preguntas: (a) ¿las top features SHAP coinciden con lo que US-018 selecciono?; (b) ¿algun feature descartado por US-018 aparece importante? (señal de refutacion); (c) ¿la dominancia AlphaEarth confirma la decision de usar el embedding como backbone?

In [15]:
# Cruce de las top SHAP con la seleccion de features de US-018.
fs_dir = Path('reports/feature_selection')
top_shap = set(
    shap_results['rf'].global_importance.head(20)['feature'].to_list()
)

fs_importance_path = fs_dir / 'feature_importance_rf.csv'
if fs_importance_path.exists():
    fs_importance = pl.read_csv(fs_importance_path)
    fs_top = set(fs_importance.head(20)['feature'].to_list())
    overlap = top_shap & fs_top
    print(f'Solapamiento top-20 SHAP vs top-20 US-018: '
          f'{len(overlap)}/20 features.')
    print('Comunes:', sorted(overlap))
    print('Solo en SHAP (revisar FE):', sorted(top_shap - fs_top))
else:
    print('reports/feature_selection/feature_importance_rf.csv '
          'no disponible — se omite el cruce cuantitativo.')

Solapamiento top-20 SHAP vs top-20 US-018: 10/20 features.
Comunes: ['EVI_fft_phase_1', 'EVI_fft_phase_2', 'EVI_fft_phase_3', 'EVI_p95', 'NDRE_p95', 'NDVI_fft_phase_1', 'NDVI_fft_phase_2', 'NDVI_p50', 'NDWI_fft_phase_1', 'PSRI_p95']
Solo en SHAP (revisar FE): ['CCCI_p75', 'EVI_p25', 'GCVI_p50', 'MCARI_p25', 'MCARI_p50', 'MCARI_p95', 'MSAVI2_min', 'NDCI_p50', 'NDRE_p50', 'NDWI_p50']


### 5.1 Hallazgos

> _Esta celda se completa al ejecutar el notebook sobre los modelos production; los numeros concretos del cruce salen de la celda anterior._

Hallazgos esperados (≥3, plan AC-5):

1. **Convergencia importance nativa vs SHAP** — los features en el top de Gini/gain y los del top SHAP coinciden en su mayoria; discrepancias señalan features con efectos no lineales o interacciones que SHAP captura mejor que la importancia nativa.
2. **Dominancia AlphaEarth** — la fraccion de dimensiones `dim_NN` en el top-20 SHAP (seccion 4.1) valida o matiza la decision irrevocable de usar AlphaEarth como backbone: si dominan, el embedding aporta la mayor parte de la senal; si no, los indices espectrales y la fenologia siguen siendo imprescindibles.
3. **Validacion del FE de US-018** — si los features que US-018 selecciono coinciden con el top SHAP, la seleccion se valida; si un feature descartado aparece arriba, se documenta como señal de refutacion en `docs/product-backlog/us-020-fe-adjustments.md`.

### 5.2 Recomendacion para el feature engineering

Si el cruce de la seccion 5 confirma la seleccion de US-018, **no se requiere ajuste**: el FE del EPIC 3 queda validado por la interpretabilidad del baseline. Si el cruce refuta alguna decision (un feature relevante descartado, o ruido retenido en el top), la recomendacion concreta se registra en `docs/product-backlog/us-020-fe-adjustments.md` para que el EPIC 5 la incorpore antes de entrenar las arquitecturas de segmentacion.

## 5b. Curvas de aprendizaje y validacion — diagnostico de sub/sobreajuste

Esta seccion diagnostica si el baseline sub o sobreajusta. Se usan dos herramientas:

- **Curva de aprendizaje**: accuracy de train y de validacion al crecer el numero de muestras de entrenamiento. Un gap grande train-val indica sobreajuste; ambas curvas bajas y juntas, subajuste.
- **Curva de validacion**: accuracy frente a un hiperparametro critico (`max_depth` para RF, `n_estimators` y `learning_rate` para XGBoost), para localizar la zona de equilibrio.

Toda la evaluacion usa el **mismo CV espacial 5-fold** (H3 + KMeans + buffer 1 km) del resto del notebook — los splits se materializan en una lista porque `learning_curve` reusa el `cv` una vez por cada tamano. El criterio de spatial CV esta documentado en `docs/spatial_cv_baseline.md`.

In [16]:
from ml.eval.learning_curves import (
    diagnose_fit,
    plot_learning_curve,
    plot_validation_curve,
)
from ml.train.baseline import _build_cv_splits

# CV espacial materializado (lista de splits posicionales).
cv_splits_5b = _build_cv_splits(
    df, k_folds=5, buffer_km=1.0, random_state=42
)
print(f'CV espacial: {len(cv_splits_5b)} folds materializados')

2026-05-22 07:50:08 [info     ] spatial_folds_cache_hit        n_folds=5 path=data\test_fixtures\baseline_spatial_folds_n85951_k5_b1_s42.parquet


CV espacial: 5 folds materializados


In [17]:
# Curva de aprendizaje RF y XGB (accuracy train/val vs n muestras).
from pathlib import Path

from ml.train.baseline import build_estimator

reports_dir = Path('reports/baseline')
reports_dir.mkdir(parents=True, exist_ok=True)
curve_train_sizes = [0.1, 0.25, 0.4, 0.55, 0.7, 0.85, 1.0]
learning_results = {}
for kind in ('rf', 'xgb'):
    estimator = build_estimator(kind, {})
    lc_result, lc_fig = plot_learning_curve(
        estimator, df, cv_splits_5b,
        train_sizes=curve_train_sizes,
        max_samples=MAX_SAMPLES,
    )
    learning_results[kind] = lc_result
    lc_fig.suptitle(f'Curva de aprendizaje — {kind.upper()}')
    lc_fig.savefig(
        reports_dir / f'learning_curve_{kind}.png',
        dpi=200, bbox_inches='tight',
    )
    plt.show()

2026-05-22 07:50:09 [info     ] learning_curve_subsampled      max_samples=3000 n_kept=2999 n_original=85951


2026-05-22 07:50:09 [info     ] learning_curve_start           n_features=185 n_folds=5 n_samples=2999 n_train_sizes=7 scoring=accuracy


2026-05-22 07:50:39 [info     ] learning_curve_done            train_acc_max=1.0 val_acc_max=0.6784


2026-05-22 07:50:39 [info     ] xgb_device_resolved            device=cuda gpu='NVIDIA GeForce RTX 4070 Laptop GPU'


2026-05-22 07:50:40 [info     ] learning_curve_subsampled      max_samples=3000 n_kept=2999 n_original=85951


2026-05-22 07:50:40 [info     ] learning_curve_start           n_features=185 n_folds=5 n_samples=2999 n_train_sizes=7 scoring=accuracy


2026-05-22 07:51:43 [info     ] learning_curve_done            train_acc_max=1.0 val_acc_max=0.6974


In [18]:
# Diagnostico explicito de sub/sobreajuste por modelo.
for kind, lc_result in learning_results.items():
    diag = diagnose_fit(lc_result)
    print(f'{kind.upper()}: veredicto={diag.verdict}  '
          f'gap={diag.gap:.4f}  '
          f'train_acc={diag.train_acc_max:.4f}  '
          f'val_acc={diag.val_acc_max:.4f}')
    print(f'  {diag.explanation}')

2026-05-22 07:51:43 [info     ] fit_diagnosed                  gap=0.3216 train_acc_max=1.0 val_acc_max=0.6784 verdict=overfit


RF: veredicto=overfit  gap=0.3216  train_acc=1.0000  val_acc=0.6784
  Sobreajuste: el gap train-val es 0.322 (> 0.10). El modelo memoriza el train (accuracy 1.000) pero generaliza peor en validacion (accuracy 0.678).
2026-05-22 07:51:43 [info     ] fit_diagnosed                  gap=0.3026 train_acc_max=1.0 val_acc_max=0.6974 verdict=overfit


XGB: veredicto=overfit  gap=0.3026  train_acc=1.0000  val_acc=0.6974
  Sobreajuste: el gap train-val es 0.303 (> 0.10). El modelo memoriza el train (accuracy 1.000) pero generaliza peor en validacion (accuracy 0.697).


In [19]:
# Curva de validacion RF — max_depth.
vc_rf, vc_rf_fig = plot_validation_curve(
    build_estimator('rf', {}), df, 'max_depth',
    [5, 10, 15, 20, 30, None], cv_splits_5b,
    max_samples=MAX_SAMPLES,
)
vc_rf_fig.suptitle('Curva de validacion — RF max_depth')
vc_rf_fig.savefig(
    reports_dir / 'validation_curve_rf_max_depth.png',
    dpi=200, bbox_inches='tight',
)
plt.show()

2026-05-22 07:51:43 [info     ] learning_curve_subsampled      max_samples=3000 n_kept=2999 n_original=85951


2026-05-22 07:51:43 [info     ] validation_curve_start         n_folds=5 n_samples=2999 n_values=6 param_name=max_depth scoring=accuracy


2026-05-22 07:52:36 [info     ] validation_curve_done          best_val_acc=0.675 param_name=max_depth


In [20]:
# Curva de validacion XGB — n_estimators.
vc_xgb_ne, vc_xgb_ne_fig = plot_validation_curve(
    build_estimator('xgb', {}), df, 'n_estimators',
    [100, 200, 300, 400, 500], cv_splits_5b,
    max_samples=MAX_SAMPLES,
)
vc_xgb_ne_fig.suptitle('Curva de validacion — XGB n_estimators')
vc_xgb_ne_fig.savefig(
    reports_dir / 'validation_curve_xgb_n_estimators.png',
    dpi=200, bbox_inches='tight',
)
plt.show()

2026-05-22 07:52:36 [info     ] xgb_device_resolved            device=cuda gpu='NVIDIA GeForce RTX 4070 Laptop GPU'


2026-05-22 07:52:36 [info     ] learning_curve_subsampled      max_samples=3000 n_kept=2999 n_original=85951


2026-05-22 07:52:36 [info     ] validation_curve_start         n_folds=5 n_samples=2999 n_values=5 param_name=n_estimators scoring=accuracy


2026-05-22 07:55:22 [info     ] validation_curve_done          best_val_acc=0.6939 param_name=n_estimators


El diagnostico `diagnose_fit` reporta un veredicto explicito (`overfit` / `underfit` / `good_fit`) con el gap train-val numerico. El baseline tabular sobre embeddings genericos tiende a un accuracy modesto: si el veredicto es `good_fit` con accuracy de validacion baja, el limite es la **capacidad del modelo**, no el sobreajuste — justificacion directa de por que el EPIC 5 incorpora arquitecturas temporales (U-TAE, TSViT) con mayor capacidad.

## 6. Desempeno minimo

El Avance 3 fija un umbral de **F1-macro >= 0.60** sobre PASTIS-R. Se entrenan RF y XGBoost con validacion cruzada **espacial** (H3 + KMeans + buffer 1 km, sin leakage entre parcelas vecinas) y se reporta la media CV de cada metrica.

La rubrica evalua que el desempeno este **declarado y justificado**, no que el umbral se supere: si F1-macro < 0.60 se documentan las causas y las decisiones para EPIC 5 en la seccion 6.1.

In [21]:
from ml.train.baseline import train_one_model, tune_baseline

results = {}
for kind in ('rf', 'xgb'):
    if TUNE:
        best_params = tune_baseline(df, model=kind)
        results[kind] = train_one_model(
            df, model=kind, hyperparams=best_params
        )
    else:
        results[kind] = train_one_model(df, model=kind)
    print(f'{kind.upper()}  entrenado.')

2026-05-22 07:55:22 [info     ] spatial_folds_cache_hit        n_folds=5 path=data\test_fixtures\baseline_spatial_folds_n85951_k5_b1_s42.parquet

2026-05-22 07:55:22 [info     ] spatial_cv_start               n_classes=18 n_folds=5


2026-05-22 07:55:22 [info     ] spatial_cv_fold_start          fold=1/5 n_test=23157 n_train=62794


2026-05-22 07:55:22 [info     ] scaler_persisted               n_features=185 n_train=62794 path=C:\Users\arthu\AppData\Local\Temp\tmplvrm1cny\fold_0_scaler.joblib version=v1


2026-05-22 07:55:34 [info     ] spatial_cv_fold_done           f1_macro=0.3895 fold=1/5


...

2026-05-22 08:01:22 [info     ] xgb_device_resolved            device=cuda gpu='NVIDIA GeForce RTX 4070 Laptop GPU'


2026-05-22 08:02:37 [info     ] spatial_cv_fold_done           f1_macro=0.3125 fold=5/5


2026-05-22 08:02:38 [info     ] xgb_device_resolved            device=cuda gpu='NVIDIA GeForce RTX 4070 Laptop GPU'


2026-05-22 08:03:51 [info     ] baseline_trained               f1_macro_oof=0.4094206066320682 model=xgb n_classes=18 n_features=185 n_samples=85951


XGB  entrenado.


In [22]:
# Tabla resumen de las metricas CV-mean por modelo.
summary = pl.DataFrame(
    [
        {
            'modelo': kind.upper(),
            **{m: round(v, 4) for m, v in res.metrics.items()},
        }
        for kind, res in results.items()
    ]
)
summary

modelo,f1_macro,f1_weighted,miou,accuracy,cohen_kappa
str,f64,f64,f64,f64,f64
"""RF""",0.365,0.6583,0.2699,0.6724,0.5961
"""XGB""",0.4094,0.6917,0.3115,0.7257,0.6546


In [23]:
# Veredicto vs el umbral del Avance 3.
best_kind = max(results, key=lambda k: results[k].metrics['f1_macro'])
best_f1 = results[best_kind].metrics['f1_macro']
passed = best_f1 >= F1_THRESHOLD
print(f'Mejor modelo: {best_kind.upper()}  |  F1-macro = {best_f1:.4f}')
print(f'Umbral Avance 3: {F1_THRESHOLD:.2f}  |  '
      f'{"ALCANZADO" if passed else "NO alcanzado — ver 6.1"}')

Mejor modelo: XGB  |  F1-macro = 0.4094
Umbral Avance 3: 0.60  |  NO alcanzado — ver 6.1


### 6.1 Causas probables y decisiones para EPIC 5

Si el F1-macro CV-mean queda por debajo de 0.60, las causas probables son:

1. **Granularidad fina de PASTIS-R (20 clases).** Varias clases de cultivo son espectralmente similares; un modelo tabular sobre un embedding anual no captura la firma fenologica que las distingue.
2. **Desbalance de clases.** Pese al balanceo (`class_weight='balanced'` en RF, `sample_weight` inverso a frecuencia en XGBoost), las clases minoritarias aportan pocas parcelas y el F1-macro las penaliza con fuerza.
3. **Limite de un modelo tabular sobre un embedding generico.** AlphaEarth resume el ano en 64 dimensiones; pierde la dinamica temporal intra-anual que un modelo de series temporales si aprovecha.

Decisiones concretas que EPIC 5 incorpora:

- **U-TAE y TSViT** explotan la serie temporal Sentinel-2 completa (no el embedding resumido), capturando la fenologia que separa cultivos similares.
- **Ensamble heterogeneo (EPIC 6)** combina el baseline tabular con los modelos temporales y un VLM, recuperando senal complementaria que ningun modelo aislado captura.

## 7. Comparativa AlphaEarth vs Sentinel-2 crudo

_Placeholder — completado por US-022 (Notebook secuencial + comparativa)._

## 8. Discusion y decisiones para EPIC 5

_Placeholder — completado por US-022 (Notebook secuencial + comparativa)._